In [1]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = r"C:\Users\dell\Desktop\Documents for UK\dissertation_project"
GOLD_DIR = os.path.join(PROJECT_ROOT, "data_gold")

path = os.path.join(GOLD_DIR, "gold_daily_kpis.csv")
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,revenue,orders,returned_orders,avg_shipping_days,aov,return_rate,total_spend_gbp,sessions,web_orders,conversion_rate_calc,cac_proxy
0,2024-01-01,12002.58,153,5,3.56,78.45,0.0327,1995.69,8616,153,0.01776,13.04
1,2024-01-02,15047.95,209,8,4.13,72.00,0.0383,2477.17,9975,209,0.02095,11.85
2,2024-01-03,9460.13,121,10,4.06,78.18,0.0826,1787.00,7447,121,0.01625,14.77
3,2024-01-04,10991.20,160,4,3.91,68.70,0.0250,1674.58,7834,160,0.02042,10.47
4,2024-01-05,5813.08,87,4,3.85,66.82,0.0460,2051.51,6976,87,0.01247,23.58


In [3]:
METRICS = {
    "revenue": "Revenue",
    "orders": "Orders",
    "aov": "AOV",
    "conversion_rate_calc": "Conversion Rate",
    "return_rate": "Return Rate",
    "cac_proxy": "CAC Proxy",
    "avg_shipping_days": "Avg Shipping Days",
}

# Ensure numeric
for col in METRICS.keys():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [4]:
#Approch 1 
def rule_based_alerts(
    df, metrics, window=30,
    z_med=2.0, z_high=3.0,
    pct_med=0.15, pct_high=0.30
):
    out = []
    work = df.copy()

    for col, name in metrics.items():
        if col not in work.columns:
            continue

        series = work[col]

        baseline = series.rolling(window=window, min_periods=max(7, window//3)).mean()
        std = series.rolling(window=window, min_periods=max(7, window//3)).std()
        z = (series - baseline) / std

        pct = series.pct_change()

        for i in range(len(work)):
            if pd.isna(work.loc[i, "date"]) or pd.isna(series.iloc[i]):
                continue

            # Need baseline ready
            if pd.isna(baseline.iloc[i]) or pd.isna(std.iloc[i]) or std.iloc[i] == 0:
                continue

            z_val = z.iloc[i]
            pct_val = pct.iloc[i]

            # severity logic
            severity = None
            reason = []

            if abs(z_val) >= z_high:
                severity = "HIGH"
                reason.append(f"|z| >= {z_high}")
            elif abs(z_val) >= z_med:
                severity = "MEDIUM"
                reason.append(f"|z| >= {z_med}")

            if pd.notna(pct_val):
                if abs(pct_val) >= pct_high:
                    severity = "HIGH"
                    reason.append(f"|pct_change| >= {pct_high:.0%}")
                elif abs(pct_val) >= pct_med and severity != "HIGH":
                    severity = "MEDIUM"
                    reason.append(f"|pct_change| >= {pct_med:.0%}")

            if severity is None:
                continue

            direction = "UP" if (series.iloc[i] - baseline.iloc[i]) > 0 else "DOWN"

            evidence = (
                f"{name}={series.iloc[i]:.4g}, baseline={baseline.iloc[i]:.4g}, "
                f"z={z_val:.2f}, pct_change={pct_val:.1%}" if pd.notna(pct_val)
                else f"{name}={series.iloc[i]:.4g}, baseline={baseline.iloc[i]:.4g}, z={z_val:.2f}"
            )

            out.append({
                "date": work.loc[i, "date"].date(),
                "metric": name,
                "value": float(series.iloc[i]),
                "baseline": float(baseline.iloc[i]),
                "z_score": float(z_val),
                "pct_change": float(pct_val) if pd.notna(pct_val) else np.nan,
                "severity": severity,
                "direction": direction,
                "reason": "; ".join(reason),
                "method": "RULES",
                "evidence": evidence,
            })

    return pd.DataFrame(out)

alerts_rules = rule_based_alerts(df, METRICS, window=30)
alerts_rules.head()

,date,metric,value,baseline,z_score,pct_change,severity,direction,reason,method,evidence
0,2024-01-10,Revenue,11880.75,11086.051000,0.233399,1.121786,HIGH,UP,|pct_change| >= 30%,RULES,"Revenue=1.188e+04, baseline=1.109e+04, z=0.23,..."
1,2024-01-12,Revenue,8188.66,10817.875000,-0.824067,-0.239349,MEDIUM,DOWN,|pct_change| >= 15%,RULES,"Revenue=8189, baseline=1.082e+04, z=-0.82, pct..."
2,2024-01-13,Revenue,9771.55,10737.388462,-0.314763,0.193303,MEDIUM,DOWN,|pct_change| >= 15%,RULES,"Revenue=9772, baseline=1.074e+04, z=-0.31, pct..."
3,2024-01-15,Revenue,14116.61,10932.233333,1.069798,0.373099,HIGH,UP,|pct_change| >= 30%,RULES,"Revenue=1.412e+04, baseline=1.093e+04, z=1.07,..."
4,2024-01-16,Revenue,11976.95,10997.528125,0.339191,-0.151570,MEDIUM,UP,|pct_change| >= 15%,RULES,"Revenue=1.198e+04, baseline=1.1e+04, z=0.34, p..."


In [5]:
rules_path = os.path.join(GOLD_DIR, "alerts_daily_rules.csv")
alerts_rules.to_csv(rules_path, index=False)
rules_path

'C:\\Users\\dell\\Desktop\\Documents for UK\\dissertation_project\\data_gold\\alerts_daily_rules.csv'

In [6]:
##Approch 2 -- ML anomaly detection (Isolation Forest)
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

feature_cols = [c for c in METRICS.keys() if c in df.columns]
X = df[feature_cols].copy()

# drop rows with missing
mask = X.notna().all(axis=1) & df["date"].notna()
X = X[mask]
dates = df.loc[mask, "date"]

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# contamination = expected % anomalies (tune 0.01–0.05)
iso = IsolationForest(
    n_estimators=300,
    contamination=0.03,
    random_state=42
)
iso.fit(X_scaled)

scores = iso.decision_function(X_scaled)  # higher = more normal
pred = iso.predict(X_scaled)  # -1 anomaly, 1 normal

ml_alerts = []
for d, p, s, row in zip(dates, pred, scores, X.to_dict("records")):
    if p == -1:
        # severity by score quantiles (lower score = more anomalous)
        ml_alerts.append({
            "date": d.date(),
            "metric": "MULTI-METRIC",
            "value": np.nan,
            "baseline": np.nan,
            "z_score": np.nan,
            "pct_change": np.nan,
            "severity": "HIGH" if s < np.quantile(scores, 0.05) else "MEDIUM",
            "direction": "N/A",
            "reason": "IsolationForest flagged anomaly",
            "method": "IFOREST",
            "evidence": f"anomaly_score={s:.4f}, snapshot={ {k: round(v,4) for k,v in row.items()} }"
        })

alerts_ml = pd.DataFrame(ml_alerts)
alerts_ml.head()

,date,metric,value,baseline,z_score,pct_change,severity,direction,reason,method,evidence
0,2024-01-03,MULTI-METRIC,NaN,NaN,NaN,NaN,HIGH,N/A,IsolationForest flagged anomaly,IFOREST,"anomaly_score=-0.0035, snapshot={'revenue': 94..."
1,2024-01-05,MULTI-METRIC,NaN,NaN,NaN,NaN,HIGH,N/A,IsolationForest flagged anomaly,IFOREST,"anomaly_score=-0.0754, snapshot={'revenue': 58..."
2,2024-01-31,MULTI-METRIC,NaN,NaN,NaN,NaN,HIGH,N/A,IsolationForest flagged anomaly,IFOREST,"anomaly_score=-0.0039, snapshot={'revenue': 84..."
3,2024-03-09,MULTI-METRIC,NaN,NaN,NaN,NaN,HIGH,N/A,IsolationForest flagged anomaly,IFOREST,"anomaly_score=-0.0197, snapshot={'revenue': 93..."
4,2024-07-18,MULTI-METRIC,NaN,NaN,NaN,NaN,HIGH,N/A,IsolationForest flagged anomaly,IFOREST,"anomaly_score=-0.0093, snapshot={'revenue': 63..."


In [7]:
ml_path = os.path.join(GOLD_DIR, "alerts_daily_ml_iforest.csv")
alerts_ml.to_csv(ml_path, index=False)
ml_path

'C:\\Users\\dell\\Desktop\\Documents for UK\\dissertation_project\\data_gold\\alerts_daily_ml_iforest.csv'

In [8]:
combined = pd.concat([alerts_rules, alerts_ml], ignore_index=True)
combined = combined.sort_values(["date", "severity", "method", "metric"], ascending=[True, False, True, True])

combined_path = os.path.join(GOLD_DIR, "alerts_daily_combined.csv")
combined.to_csv(combined_path, index=False)

combined_path

'C:\\Users\\dell\\Desktop\\Documents for UK\\dissertation_project\\data_gold\\alerts_daily_combined.csv'